In [1]:
import k_fail_MBTR
import BB_wrapper
import k_fail_prediction

import torch

/Users/karim/projects/k-sPSS/.venv/lib/python3.11/site-packages/pycutest/__init__.py:26: RuntimeWarning: the PYCUTEST_CACHE environment variable is not set; current folder will be used for caching.
  warnings.warn("the PYCUTEST_CACHE environment variable is not set; current folder will be used for caching.", RuntimeWarning)


# Wrapper

In [2]:
cutest_wrapper = BB_wrapper.BB_cutest_collection(write_to_file="cutest_problem_selection.txt", max_dim=100, cap_n_problems=1)

loading problems...
0/276
loaded 1 problems


In [3]:
# evaluating n problem functions
for i in range(1):
    p = cutest_wrapper.problems[i]
    f = cutest_wrapper.problem_functions[i]
    print(f"{p.name} | n: {p.n}: {f(torch.from_numpy(p.x0))}")

SISSER | n: 2: 3.02030030003


# MBTR

In [4]:
problem_idx = 0
k = 2
k_fail_wrapper = BB_wrapper.BB_k_fail_wrapper(cutest_wrapper.problem_functions[problem_idx], k*torch.ones((1024, 2), dtype=torch.int16), 8)

alg = k_fail_MBTR.MBTR_k_fail(torch.from_numpy(cutest_wrapper.problems[problem_idx].x0), k_fail_wrapper, 1, 1e-1, 0.1, 1 +0*0.5, 1e-1, k_fail_prediction.constant_prediction_software(k), log_file_path="alg_logs/MBTR/testing.txt")

In [5]:
alg.log_current()
for i in range(1000):
    if alg.step_default():
        break


# is n + 2k + 1 fair?
# with it, always linear: 1847 function evals
# with always quad: 448 function evals

P: 7
raw points tensor([[ 1.6775,  0.8355],
        [ 0.7529,  1.0690],
        [ 0.2644,  0.7774],
        [ 0.3618,  0.8699],
        [ 1.7686, -0.5397],
        [ 1.9776,  0.3105],
        [ 1.9836,  0.2803]], dtype=torch.float64)
completed idx tensor([1, 5, 2, 6, 4], dtype=torch.int32)
x_k tensor([1.0000, 0.1000], dtype=torch.float64)
f(x_k) 3.02030030003
POINTS: tensor([[ 1.0000,  0.1000],
        [ 0.7529,  1.0690],
        [ 1.9776,  0.3105],
        [ 0.2644,  0.7774],
        [ 1.9836,  0.2803],
        [ 1.7686, -0.5397]], dtype=torch.float64)
func vals tensor([ 3.0203,  6.1768, 46.6669,  1.1952, 47.0827, 31.4318])
GOT: tensor([[ 1.0000,  0.1000],
        [ 0.7529,  1.0690],
        [ 1.9776,  0.3105],
        [ 0.2644,  0.7774],
        [ 1.9836,  0.2803],
        [ 1.7686, -0.5397]], dtype=torch.float64)
TRIMMED: tensor([[ 1.0000,  0.1000],
        [ 0.7529,  1.0690],
        [ 1.9776,  0.3105],
        [ 0.2644,  0.7774],
        [ 1.9836,  0.2803],
        [ 1.7686, -0.53

DCPError: Problem does not follow DCP rules. Specifically:
The objective is not DCP. Its following subexpressions are not:
QuadForm(var197, [[30.76 -12.39]
 [-12.39 -10.43]])

In [7]:
# H: None
# points: [[ 0.          0.        ]
#  [-0.15244442  0.4307067 ]
#  [-0.05507779  0.22729501]
#  [-0.39858192  0.6989344 ]
#  [-0.04768276  0.20510971]
#  [-0.0833317   0.29964896]]

# fixed version (still breaks)
# H: None
# points: [[0.         0.        ]
#  [0.21817741 0.44988734]
#  [0.45346695 0.21063653]
#  [0.45938012 0.19740793]
#  [0.41485953 0.27909064]
#  [0.49603891 0.06281256]]


broken_x_k = torch.tensor([1.0000, 0.1000])
broken_points = torch.tensor([
 [0.        , 0.        ],
 [0.21817741, 0.44988734],
 [0.45346695, 0.21063653],
 [0.45938012, 0.19740793],
 [0.41485953, 0.27909064],
 [0.49603891, 0.06281256]])

broken_func_vals = []

for p in broken_points:
    print(p, " | ", cutest_wrapper.problem_functions[0](p + broken_x_k))
    broken_func_vals.append(cutest_wrapper.problem_functions[0](p + broken_x_k))

broken_func_vals = torch.tensor(broken_func_vals)


tensor([0., 0.])  |  3.0203003006439277
tensor([0.2182, 0.4499])  |  7.77809909471623
tensor([0.4535, 0.2106])  |  13.824444939862795
tensor([0.4594, 0.1974])  |  14.008260278544622
tensor([0.4149, 0.2791])  |  12.659262761617814
tensor([0.4960, 0.0628])  |  15.148476370717711


In [2]:
import models
import torch

models.get_quad_model_and_solution(torch.rand(7, 2), torch.rand(7), 1)

(tensor([-0.2665,  0.6287], dtype=torch.float64),
 -1.0173285903113656,
 tensor([2.1055, 1.2922], dtype=torch.float64),
 <function models.get_quad_model_and_solution.<locals>.f_tilda(x)>)

In [6]:
broken_func_vals

tensor([0.0000, 0.1490, 0.1510, 0.1546, 0.1339, 0.1836])

In [10]:
torch.cat((torch.tensor(1).unsqueeze(0), torch.rand(5)))

tensor([1.0000, 0.1447, 0.5315, 0.1587, 0.6542, 0.3278])

In [ ]:
torch.rand(5).ap